# cycle-detection-temp-set — worked example 3: Reject a cyclic tensor-op dependency graph before execution

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A computation graph is a set of ops, each depending on the ops that produce its inputs. Before running backward you must verify the graph is acyclic, otherwise scheduling never terminates. The check is the same temp-set DFS: `temp` = ops currently being resolved (on the stack), `perm` = ops whose whole dependency subtree is verified. Re-entering a `temp` op is a back-edge — a cyclic dependency.

## Worked solution

Each op `i` has a list `deps[i]` of op indices it depends on, and a tensor it would produce. We must return the safe execution order (deps before dependents) or raise if a dependency cycle exists, then actually run a toy forward pass to prove the order is usable.

**Step 1 — set up state.** `perm` and `temp` hold op indices. `order` collects ops in finish order.

**Step 2 — DFS with the gray check.** `visit(i)`: return if `i in perm`; raise `ValueError` if `i in temp` (a dependency loop — op i transitively depends on itself). Otherwise add `i` to `temp`, resolve every dependency first, then move `i` from `temp` to `perm` and append it. Appending after dependencies means `order` already lists producers before consumers — no final reversal needed because we recursed into *dependencies* (inputs) rather than successors.

**Step 3 — execute in that order.** We give op 0 a fixed seeded tensor and define each later op as the elementwise sum of its dependencies' outputs (via `torch.stack(...).sum(0)`). Walking `order` guarantees every dependency tensor already exists when we compute an op.

**Why it's correct.** The temp set is precisely the chain of ops we are mid-way through resolving; re-entering it means an op depends on itself through some path, which is a cycle and rightly raises. Because a vertex is emitted only once all its dependencies are emitted, the resulting `order` is a valid dependency-respecting schedule.

In [ ]:
import torch as t

def schedule(deps):
    perm, temp, order = set(), set(), []

    def visit(i):
        if i in perm:
            return
        if i in temp:
            raise ValueError(f"cyclic dependency at op {i}")
        temp.add(i)
        for d in deps[i]:
            visit(d)
        temp.remove(i)
        perm.add(i)
        order.append(i)

    for i in range(len(deps)):
        visit(i)
    return order


# op0 has no deps; op1 needs op0; op2 needs op0+op1; op3 needs op2
deps = [[], [0], [0, 1], [2]]
order = schedule(deps)
print("exec order:", order)

t.manual_seed(0)
out = {}
for i in order:
    if not deps[i]:
        out[i] = t.randn(3)
    else:
        out[i] = t.stack([out[d] for d in deps[i]]).sum(0)
print("op3 output:", out[3])

try:
    schedule([[1], [0]])
except ValueError as e:
    print("raised:", e)